### Q1. Embedding a query
Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

- -0.31  
- -0.02  
- 0.12  
- 0.44

In [89]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

# Encode the query.
qv = embed.encode(q)

print(f"The first value of the vector (v[0]): {qv[0]}.")

The first value of the vector (v[0]): -0.02058203437252893.


### Setup for Q2

In [9]:
# Load the lesson pages from the course repository, pinned to commit 8c1834d
# This means everyone works with the same 72 markdown pages.
# Each document includes a filename and content, which we'll use for embedding and search.

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()] #List comprehension of homework 1 step.

print(len(documents))
print(documents[0]["filename"])
print(documents[0]["content"][:100])

72
01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxU


### Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

- 0.07
- 0.37
- 0.68
- 0.92

In [8]:
target_index, target = next(
    (i, doc)
    for i, doc in enumerate(documents)
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

v_doc = embed.encode(target["content"])

print(target_index)
print(target["filename"])
print(target["content"][:100])

22
02-vector-search/lessons/07-sqlitesearch-vector.md
# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKes


In [88]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
d = documents[22]["content"]

# Encode the query and the document.
qv = embed.encode(q)
dv = embed.encode(d)

# Compare the similarity scores.
score = qv.dot(dv)

print(f"Cosine similarity of Module 2 Lesson 7 content with the query vector from Q1 [Similarity(q, d)]: {score}.")

Cosine similarity of Module 2 Lesson 7 content with the query vector from Q1 [Similarity(q, d)]: 0.36107026789538205.


### Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

>```
>from gitsource import chunk_documents
>chunks = chunk_documents(documents, size=2000, step=1000)
>```

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

>```
>scores = X.dot(v)
>```

Which file does the highest-scoring chunk belong to (its filename)?

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- 02-vector-search/lessons/07-sqlitesearch-vector.md
- 02-vector-search/lessons/09-onnx-embedder.md

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000) # documents is the variable we set above

print(f"{chunks[0]['filename']}\n{chunks[1]['filename']}")

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/01-intro.md


In [20]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk["content"] for chunk in chunks[i:i + batch_size]] # chunks are dicts, so extract the content field instead of passing the whole list item. In the lesson, we converted into a string first.
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

print(f"Created embedding matrix with shape: {X.shape}")

  0%|          | 0/6 [00:00<?, ?it/s]

Created embedding matrix with shape: (295, 384)


In [87]:
q = "How does approximate nearest neighbor search work?"
qv = embed.encode(q)

scores = X.dot(qv)
idx = np.argmax(scores)

print(f"Query encoded successfully with shape: {qv.shape}")
print(f"Best matching chunk index: {idx}")
print(f"The highest-scoring chunk belongs to: {chunks[idx]['filename']}.")

Query encoded successfully with shape: (384,)
Best matching chunk index: 94
The highest-scoring chunk belongs to: 02-vector-search/lessons/07-sqlitesearch-vector.md.


### Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- 04-evaluation/lessons/05-search-metrics.md
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

In [35]:
"""
======================================================================
Vector Search with minsearch
======================================================================
Use minsearch to store document embeddings in memory and search them later:
- fit builds the in-memory vector index from embeddings and documents
- search compares a query vector against stored vectors
- this is a simple first step before using a persistent vector database
"""
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

print(f"Indexed {len(chunks)} chunks with {X.shape[1]}-dimensional vectors")
print("We pass the NumPy array X with all embeddings and the list of chunks as payload.")

Indexed 295 chunks with 384-dimensional vectors
We pass the NumPy array X with all embeddings and the list of chunks as payload.


In [86]:
"""
======================================================================
Searching
======================================================================
Search the vector index with a query embedding:
- encode the query into a vector first
- search compares the query vector against all stored chunk vectors
- minsearch returns results in best-first order
- results[0] is the top-scoring match
"""
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
top_result = results[0]

print(f"Query: {query}")
print(f"Query vector shape: {query_vector.shape}")
print(f"Number of results: {len(results)}")
print(f"Top Result: {top_result['filename']}.")

Query: What metric do we use to evaluate a search engine?
Query vector shape: (384,)
Number of results: 5
Top Result: 04-evaluation/lessons/05-search-metrics.md.


### Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

- 02-vector-search/lessons/01-intro.md
- 02-vector-search/lessons/02-embeddings.md
- 02-vector-search/lessons/08-pgvector.md
- 03-orchestration/lessons/05-rag.md

In [90]:
query = "How do I store vectors in PostgreSQL?"

# Text search
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

def search(query):
    return index.search(
        query,
        num_results=5
    )

text_search_results = search(query)

text_search_files = []

for ts_result in text_search_results: 
    text_search_files.append(ts_result['filename'])
    
print("Text Search:")
for i, ts_result in enumerate(text_search_results, start=1):
    print(f"{i}. {ts_result['filename']}")

# Vector search
query_vector = embed.encode(query)

vector_search_results = vindex.search(query_vector, num_results=5)

vector_search_files = []

for vs_result in vector_search_results: 
    vector_search_files.append(vs_result['filename'])

print("\nVector Search:")
for i, vs_result in enumerate(vector_search_results, start=1):
    print(f"{i}. {vs_result['filename']}")

diff = tuple(set(vector_search_files) - set(text_search_files))

print(f"\nThe file that shows up in the vector results but not in the text results is {diff[0]}.")

Text Search:
1. 02-vector-search/lessons/02-embeddings.md
2. 03-orchestration/lessons/05-rag.md
3. 02-vector-search/lessons/01-intro.md
4. 03-orchestration/lessons/05-rag.md
5. 02-vector-search/lessons/01-intro.md

Vector Search:
1. 02-vector-search/lessons/08-pgvector.md
2. 02-vector-search/lessons/08-pgvector.md
3. 03-orchestration/lessons/05-rag.md
4. 02-vector-search/lessons/08-pgvector.md
5. 02-vector-search/lessons/08-pgvector.md

The file that shows up in the vector results but not in the text results is 02-vector-search/lessons/08-pgvector.md.


### Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

>```
>RRF(d) = sum over lists of  1 / (k + rank(d))
>```

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

>```
>def rrf(result_lists, k=60, num_results=5):
>    scores = {}
>    docs = {}
>
>    for results in result_lists:
>        for rank, doc in enumerate(results):
>            key = (doc["filename"], doc["start"])
>            scores[key] = scores.get(key, 0) + 1 / (k + rank)
>            docs[key] = doc
>
>    ranked = sorted(scores, key=scores.get, reverse=True)
>    return [docs[key] for key in ranked[:num_results]]
>```

Now run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

>```
>results = rrf([vector_results, text_results])
>```

Which file is ranked first after RRF?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md

Notice that this file isn't first in either search on its own - it wins because it ranks high in both.

In [92]:
"""
======================================================================
Reciprocal Rank Fusion (RRF)
======================================================================
Args:
    result_lists: A list of ranked result lists.
    k: Rank smoothing constant, usually 60.
    num_results: Number of final results to return.

Returns:
    The top fused results after combining all ranked lists.
"""

def rrf(result_lists, k=60, num_results=5):
    # Store the accumulated fusion score for each unique chunk.
    scores = {}
    # Keep the actual chunk objects so we can return them later.
    docs = {}

    # Assumes result_lists is a list of ranked search result lists.
    # Each inner list contains chunk dictionaries with filename and start keys.
    for results in result_lists:
        # rank is the position of the chunk in this list, starting at 0.
        for rank, doc in enumerate(results):
            # Use filename + start to uniquely identify the chunk.
            # This lets the same chunk be matched across different search methods.
            key = (doc["filename"], doc["start"])

            # Add a rank-based score contribution from this list.
            # Higher-ranked chunks get a larger contribution.
            scores[key] = scores.get(key, 0) + 1 / (k + rank)

            # Save the chunk so it can be returned after sorting.
            docs[key] = doc

    # Sort chunks by their total fused score, highest first.
    ranked = sorted(scores, key=scores.get, reverse=True)

    # Return the top num_results fused chunks.
    return [docs[key] for key in ranked[:num_results]]

print("Defined Reciprocal Rank Fusion (RRF) for hybrid search.")

Defined Reciprocal Rank Fusion (RRF) for hybrid search.


In [97]:
# Combine vector and text search results using Reciprocal Rank Fusion (RRF) for "How do I give the model access to tools?"

query = "How do I give the model access to tools?"

# Text search
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

def search(query):
    return index.search(
        query,
        num_results=5
    )

text_search_results = search(query)

text_search_files = []

for ts_result in text_search_results: 
    text_search_files.append(ts_result['filename'])

# Vector search
query_vector = embed.encode(query)

vector_search_results = vindex.search(query_vector, num_results=5)

vector_search_files = []

for vs_result in vector_search_results: 
    vector_search_files.append(vs_result['filename'])

rrf_results = rrf([vector_search_results, text_search_results])

print("\nReciprocal Rank Fusion (RRF):")
for i, result in enumerate(rrf_results, start=1):
    print(f"{i}. {result['filename']}")

print(f"\nTop Ranked file after Reciprocal Rank Fusion (RRF): {rrf_results[0]['filename']}")


Reciprocal Rank Fusion (RRF):
1. 01-agentic-rag/lessons/13-function-calling.md
2. 01-agentic-rag/lessons/01-intro.md
3. 01-agentic-rag/lessons/14-agentic-loop.md
4. 04-evaluation/lessons/02-ground-truth.md
5. 01-agentic-rag/lessons/16-other-frameworks.md

Top Ranked file after Reciprocal Rank Fusion (RRF): 01-agentic-rag/lessons/13-function-calling.md
